# Fake News Detector
Run top to bottom. Needs `pip install -r requirements-train.txt`. The deployable version of this logic lives in `train.py` / `app.py`.

In [ ]:
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from preprocess import clean_text

## Load data

In [ ]:
fake = pd.read_csv('data/Fake.csv')
true = pd.read_csv('data/True.csv')
fake['label'] = 0   # fake
true['label'] = 1   # real
df = pd.concat([fake, true], ignore_index=True)
df.shape

## Clean
Real articles start with a `CITY (Reuters) -` tag (99% of them, 0% of the fake ones), which lets the model cheat. `clean_text` strips it. Duplicates are removed so the same article can't land in both train and test.

In [ ]:
df['content'] = (df['title'].fillna('') + ' ' + df['text'].fillna('')).map(clean_text)
df = df[df['content'].str.len() > 20]
df = df.drop_duplicates(subset='content').reset_index(drop=True)
df['label'].value_counts()

## Train

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['content'], df['label'], test_size=0.2, random_state=42, stratify=df['label'])

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=50000, ngram_range=(1, 2), min_df=3, sublinear_tf=True)),
    ('clf', LogisticRegression(max_iter=1000, C=5)),
])
pipe.fit(X_train, y_train)

## Evaluate

In [ ]:
pred = pipe.predict(X_test)
print('Accuracy:', accuracy_score(y_test, pred))
print(classification_report(y_test, pred, target_names=['fake', 'real']))

## Save (one file: vectorizer + model together)

In [ ]:
joblib.dump(pipe, 'model.joblib', compress=3)